# SORT1 rs12740374 — ChromBPNet effect vs. cross-species conservation

Reproduces the MCP `analyze_variant_multilayer` call for the
ChromBPNet oracle, which loads one model per (assay, cell_type)
pair rather than taking a generic track list.


In [ ]:
# All imports the notebook needs — top-level, no later imports.
import json
from pathlib import Path

import chorus
from chorus.analysis.normalization import get_normalizer
from chorus.analysis.variant_report import build_variant_report
from chorus.analysis.analysis_request import AnalysisRequest

In [ ]:
WALKTHROUGH_DIR = Path.cwd()
print(f"Writing artifacts to: {WALKTHROUGH_DIR}")

In [ ]:
# Create the ChromBPNet oracle and load a specific assay+cell_type model.
oracle = chorus.create_oracle(
    oracle_name='chrombpnet',
    use_environment=True,
)
oracle.load_pretrained_model(
    assay='DNASE',
    cell_type='HepG2',
    fold=0,
)

In [ ]:
# Inputs.
oracle_name = 'chrombpnet'
position = 'chr1:109274968'
ref_allele = 'G'
alt_alleles = ['T']
gene_name = 'SORT1'

In [ ]:
# ChromBPNet works on a narrow ±100 bp prediction interval around the
# variant (the model's input width is 2114 bp; we keep the user-facing
# region small and let the oracle extend internally).
_chrom, _pos = position.split(":")
_pos = int(_pos)
region = f"{_chrom}:{_pos - 100}-{_pos + 100}"

In [ ]:
# Score the variant. assay_ids=[] tells the loaded ChromBPNet model to
# return its single track.
variant_result = oracle.predict_variant_effect(
    genomic_region=region,
    variant_position=position,
    alleles=[ref_allele] + alt_alleles,
    assay_ids=[],
)

In [ ]:
# Build the report with the ChromBPNet per-track normalizer.
normalizer = get_normalizer(oracle_name=oracle_name)
analysis_request = AnalysisRequest(
    user_prompt=f"Score {position} {ref_allele}>{alt_alleles[0]} "
                f"using ChromBPNet 'DNASE' model in "
                f"'HepG2', overlay conservation tracks.",
    tool_name="analyze_variant_multilayer",
    oracle_name='chrombpnet',
    tracks_requested="DNASE:HepG2",
)
report = build_variant_report(
    variant_result=variant_result,
    oracle_name=oracle_name,
    gene_name=gene_name,
    normalizer=normalizer,
    igv_raw=False,
    analysis_request=analysis_request,
    show_conservation=True,
)

In [ ]:
# Real cross-species conservation at the variant's own base -- not part of
# the oracle's effect scores, so it is measured directly and merged into the
# saved JSON below rather than left as an unbacked claim in the README.
from chorus.analysis.conservation import (
    read_entropy_values, read_phastcons_values, read_phylop_values,
)

conservation_scores = {
    "phyloP_100way": float(read_phylop_values(_chrom, _pos - 1, _pos)[0]),
    "phastCons_100way": float(read_phastcons_values(_chrom, _pos - 1, _pos)[0]),
    "gpn_star_entropy_bits": float(read_entropy_values(_chrom, _pos - 1, _pos)[0]),
}
print(conservation_scores)

In [ ]:
# Save the same artifacts the MCP tool would produce:
#   - example_output.md  (markdown report)
#   - example_output.json (structured scores)
#   - example_output.tsv (track-level table)
#   - rs12740374_SORT1_chrombpnet_conservation_report.html (interactive IGV report)
_report_dict = report.to_dict()
_report_dict["conservation"] = conservation_scores
WALKTHROUGH_DIR.joinpath("example_output.md").write_text(report.to_markdown())
WALKTHROUGH_DIR.joinpath("example_output.json").write_text(
    json.dumps(_report_dict, indent=2, default=str)
)
try:
    report.to_dataframe().to_csv(
        WALKTHROUGH_DIR / "example_output.tsv", sep="\t", index=False,
    )
except Exception as exc:
    print(f"TSV write skipped: {exc}")

_html_path = report.to_html(output_path=str(WALKTHROUGH_DIR / "rs12740374_SORT1_chrombpnet_conservation_report.html"))
print(f"Wrote artifacts to { WALKTHROUGH_DIR }")


## What this notebook produced

- `example_output.md` — markdown report (1bp-resolution ChromBPNet profile)
- `example_output.json` — structured per-bin scores plus a `conservation` block (phyloP, phastCons, GPN-Star entropy at the variant)
- `example_output.tsv` — track table
- `rs12740374_SORT1_chrombpnet_conservation_report.html` — IGV browser with 1bp ref / alt profile overlay and conservation tracks
